# Tesla Model 3 Performance — lap time telemetry

This notebook reads Tesla Track Mode telemetry from a configurable directory. Use the CSV dropdown to change sessions and the lap dropdown to inspect any recorded timed lap.

The synchronized explorer places speed, driver inputs, vehicle dynamics, power, and steering graphs above the GPS track. Drag the yellow position handle on the track (or click the trace / drag the time slider) to move every graph to the corresponding recorded sample.

## Environment

The draggable map uses the Jupyter Matplotlib widget backend. From the repository root, install the dependencies once with <code>python -m pip install -r requirements.txt</code>, then select the <code>car_track</code> kernel and restart it.

The live dashboard cannot be restored from a stale saved widget output. Use **Restart Kernel and Run All** after reopening this notebook.

In [1]:
from importlib.util import find_spec
from pathlib import Path
import sys

EXPECTED_PYTHON = Path(r"C:\Users\11320\anaconda3\envs\car_track\python.exe")
if Path(sys.executable).resolve() != EXPECTED_PYTHON.resolve():
    raise RuntimeError(
        f"Wrong notebook kernel: {sys.executable}. Select the car_track environment at "
        f"{EXPECTED_PYTHON}, restart the kernel, and run all cells again."
    )

if find_spec("ipympl") is None:
    raise ModuleNotFoundError(
        "The draggable track view needs ipympl. Run %pip install -r requirements.txt, "
        "restart the kernel, and then run the notebook again."
    )

get_ipython().run_line_magic("matplotlib", "widget")

from IPython.display import display

from telemetry_utils import (
    TelemetryDashboard,
    build_sector_summary,
    format_lap_time,
)


## Configuration

Change <code>DATA_DIR</code> to point at another folder. The refresh button rescans CSV files directly inside that folder. Leave <code>DEFAULT_FILE</code> as <code>None</code> to open the newest filename, or set it to a CSV filename.

In [2]:
DATA_DIR = Path("data/20260714")
N_SECTORS = 3
DEFAULT_FILE = None  # Example: "telemetry-v1-2026-07-14-18_44_10.csv"


## Interactive lap explorer

1. Choose a telemetry CSV. Files that contain only Lap 0 remain selectable and show a clear empty state.
2. Choose a timed lap and preferred speed unit.
3. Drag the yellow handle along the GPS trace. It snaps to the nearest Tesla sample and updates the time cursor, graph markers, slider, and exact telemetry readout.

The session summary, fastest lap, equal-distance sector estimate, lap table, and plots all refresh when the selected CSV changes.

In [3]:
previous_dashboard = globals().get("dashboard")
if previous_dashboard is not None:
    previous_dashboard._clear_figure()
    previous_dashboard.widget.close()

dashboard = TelemetryDashboard(
    DATA_DIR,
    n_sectors=N_SECTORS,
    default_file=DEFAULT_FILE,
)
dashboard.display()


## Optional programmatic access

The dashboard keeps its currently loaded frames available for further analysis. After changing a dropdown, rerun the next cell if you want fresh local variable names.

In [4]:
selected_session = dashboard.telemetry
timed = dashboard.timed
lap_summary = dashboard.lap_summary
selected_lap_data = dashboard.lap_data

display(lap_summary)

if not timed.empty:
    sector_summary, best_sectors = build_sector_summary(timed, N_SECTORS)
    theoretical_ms = best_sectors["Sector Time (ms)"].sum()
    print(f"Theoretical {N_SECTORS}-sector lap: {format_lap_time(theoretical_ms)}")
    display(sector_summary)
    display(best_sectors)


,Lap,Lap Time (ms),Lap Time,Gap to Fastest (s),Samples,Average Speed (MPH),Maximum Speed (MPH),Average Speed (km/h),Maximum Speed (km/h),Average Throttle (%),Maximum Brake (bar),Median Sample Interval (ms)
0,1,115082,1:55.082,0.000,5755,75.762909,137.944405,121.928584,222.000001,51.163580,76.200004,17.0
1,2,116182,1:56.182,1.100,5809,75.006103,137.323033,120.710621,220.999999,53.232571,82.200004,17.0
2,3,121799,2:01.799,6.717,6089,71.474933,128.002466,115.027755,206.000001,57.086484,75.300004,17.0
3,4,121889,2:01.889,6.807,6094,71.441271,117.439155,114.973581,188.999999,52.841353,52.200003,17.0
4,5,236773,3:56.773,121.691,11835,38.381261,96.312535,61.768652,155.000000,21.381225,15.000002,17.0


Theoretical 3-sector lap: 1:54.908


,Lap,S1,S2,S3,Sector Sum,GPS Distance (m)
0,1,37160.886250,43737.032399,34184.081351,115082.0,3911.116832
1,2,37740.843886,43563.314398,34877.841715,116182.0,3909.488159
2,3,39279.293210,46143.579149,36376.127642,121799.0,3894.009763
3,4,38938.995931,45538.946245,37411.057824,121889.0,3900.700201
4,5,57355.959020,67932.236518,111484.804462,236773.0,4010.379341


,Sector,Lap,Sector Time (ms)
0,1,1,37160.886250
1,2,2,43563.314398
2,3,1,34184.081351


## Interpretation notes

- Lap 0 is session/out-lap data in these recordings because its elapsed time remains zero, so it is excluded from timed-lap controls.
- A lap time is the maximum recorded elapsed time for that positive lap ID. The notebook does not guess whether a long final lap was interrupted or became an in-lap.
- Sector boundaries are equal fractions of measured GPS distance; GPS noise and racing-line differences can shift equivalent boundaries slightly between laps.
- Negative brake sensor offsets are preserved in the loaded data and clipped to zero only in the brake plot/readout.
- The track is drawn directly from GPS coordinates without a basemap, keeping the explorer offline and reproducible.